# Fase 1: Preprocesamiento y PCA

En este notebook nos encargaremos de limpiar los datos, escalarlos usando `StandardScaler` y realizar un Análisis de Componentes Principales (PCA) preliminar para visualizar las distribuciones espaciales de los cultivos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Carga de Datos y Limpieza Básica

In [ ]:
data_path = "../data/raw/sensor_Crop_Dataset.csv"
df = pd.read_csv(data_path)
print(f"Dimensiones iniciales: {df.shape}")

# Eliminar filas duplicadas
df = df.drop_duplicates()

# Tratar valores nulos (si los hay)
# Dependiendo del análisis EDA, podemos eliminarlos o imputarlos. 
# Aquí los eliminamos para tener un dataset limpio base.
df = df.dropna()

print(f"Dimensiones tras limpieza: {df.shape}")
df.head()

## 2. Escalado de Variables
Utilizaremos `StandardScaler` ya que, según el EDA, la distribución no presentó atípicos extremadamente severos que requirieran obligatoriamente un `RobustScaler`.

In [ ]:
# Separamos variables predictoras y variable objetivo
# (Asumimos que 'Crop' es la etiqueta de validación y las demás son numéricas)
target = 'Crop'

# Identificar numéricas excluyendo categóricas como Soil_Type si existiera
num_cols = df.select_dtypes(include=np.number).columns.tolist()

X = df[num_cols]
y = df[target] if target in df.columns else None

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertimos de nuevo a DataFrame para mejor manejo
df_scaled = pd.DataFrame(X_scaled, columns=num_cols)
if y is not None:
    df_scaled[target] = y.values
    
df_scaled.head()

## 3. Análisis de Componentes Principales (PCA)
Calculamos el PCA para reducir la dimensionalidad y entender cuánta varianza explican los primeros componentes.

In [ ]:
pca = PCA()
pca.fit(X_scaled)

explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.title('Varianza Acumulada Explicada por Componentes Principales')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Acumulada')
plt.axhline(y=0.80, color='r', linestyle='-')
plt.text(1, 0.82, '80% Varianza', color='red', fontsize=12)
plt.grid(True)
plt.show()

## 4. Visualización de Clústeres Naturales (PCA 2D y 3D)

In [ ]:
# Obtenemos los 3 primeros componentes para graficar
pca_3 = PCA(n_components=3)
components = pca_3.fit_transform(X_scaled)

df_pca = pd.DataFrame(data=components, columns=['PC1', 'PC2', 'PC3'])
if y is not None:
    df_pca[target] = y.values

# Gráfica 2D
plt.figure(figsize=(10, 8))
if y is not None:
    sns.scatterplot(x='PC1', y='PC2', hue=target, data=df_pca, palette='tab20', alpha=0.7)
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)
else:
    sns.scatterplot(x='PC1', y='PC2', data=df_pca, alpha=0.7)
plt.title('Visualización PCA 2D')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Gráfica 3D (estática)
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

if y is not None:
    crops = df_pca[target].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(crops)))
    
    for i, crop in enumerate(crops):
        subset = df_pca[df_pca[target] == crop]
        ax.scatter(subset['PC1'], subset['PC2'], subset['PC3'], label=crop, color=colors[i], alpha=0.6)
else:
    ax.scatter(df_pca['PC1'], df_pca['PC2'], df_pca['PC3'], alpha=0.6)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
plt.title('Visualización PCA 3D')
plt.show()

## 5. Exportar Dataset Preprocesado
Guardamos el dataset escalado (que servirá de entrada al modelo de K-Means).

In [ ]:
processed_path = "../data/processed/sensor_Crop_Dataset_scaled.csv"
df_scaled.to_csv(processed_path, index=False)
print(f"Dataset guardado exitosamente en {processed_path}")